# TFI 1: Mejora y restauracion de imagenes

**Modalidad:** trabajo en parejas con defensa oral breve e individual.

**Entrega:** notebook completo + carpeta de imagenes originales + carpeta de resultados procesados.

En este Trabajo Final Integrador vas a reunir tecnicas vistas en las unidades de bibliotecas basicas de PDI y vision por computadora para resolver tres casos obligatorios. La idea no es aplicar filtros de manera automatica, sino leer un problema visual, comparar estrategias y justificar una decision tecnica.

Los tres casos son:

1. imagen obtenida mediante camara oscura;
2. imagen de medio grafico color;
3. imagen de medio grafico blanco/negro.


## Objetivo

Construir tres pipelines acotados de mejora y restauracion, comparar alternativas por caso y sostener una eleccion final con evidencia visual y criterio tecnico.

## Resultados de aprendizaje esperados

Al finalizar este trabajo, vas a poder:

- diagnosticar problemas frecuentes de contraste, ruido, perspectiva e iluminacion;
- elegir operaciones pertinentes segun el tipo de imagen;
- justificar parametros y descartar estrategias poco adecuadas;
- comparar resultados sin reducir la decision a "queda mejor";
- explicar limites tecnicos de una restauracion digital.

## Relacion con la secuencia

Este TFI recupera y combina ideas trabajadas en `003 - librerias_fundamentos_pdi` y `004 - computer_vision_parte_1`: diagnostico inicial, ajuste de brillo y contraste, `CLAHE`, cambio de perspectiva, suavizado, umbralizacion, morfologia e `inpainting`.


## Trabajo con IA

Podes usar IA como apoyo para discutir alternativas, revisar errores de codigo o auditar una decision tecnica. Pero no vale usarla como reemplazo de tu criterio. Toda seleccion de parametros, toda justificacion y toda version entregada tienen que quedar bajo tu responsabilidad.

### Registro breve del uso de IA

Completa al menos una entrada por caso.

| Caso | Objetivo de la consulta | Pedido a la IA | Que conservaste y por que | Que descartaste y por que | Que aprendiste |
|---|---|---|---|---|---|
| Camara oscura | | | | | |
| Medio grafico color | | | | | |
| Medio grafico blanco/negro | | | | | |


## Restricciones del trabajo

- Tenes que resolver los **tres casos obligatorios**.
- En cada caso tenes que mostrar al menos **dos estrategias comparables**.
- Cada estrategia tiene que responder a un problema visual observado, no a una receta previa.
- No alcanza con decir que una imagen "queda mejor". Tenes que explicar **que problema resolvio**, **que cambio produjo** y **que limite mantiene**.
- Conviene que cada pipeline tenga entre **2 y 4 operaciones principales**. Si agregas mas pasos, tenes que justificar por que cada uno aporta algo distinto.
- Para el caso blanco/negro, si necesitas apoyo, revisa antes o durante este trabajo `004 - computer_vision_parte_1/006b2 - umbralizacion global, Otsu y adaptive threshold.ipynb`.


## Modulos que vamos a usar

- `pathlib.Path`: para trabajar con rutas de archivo.
- `cv2`: para leer imagenes y aplicar operaciones de procesamiento.
- `numpy`: para operar con arreglos y mascaras.
- `matplotlib.pyplot`: para comparar visualmente resultados.
- `pandas`: para construir la tabla final de comparacion.


In [ ]:
# Importamos herramientas de rutas para manejar archivos de entrada y salida.
from pathlib import Path

# Importamos OpenCV para cargar y transformar imagenes.
import cv2

# Importamos NumPy para trabajar con matrices y mascaras.
import numpy as np

# Importamos Pandas para construir la tabla final del trabajo.
import pandas as pd

# Importamos Matplotlib para mostrar comparaciones dentro del notebook.
import matplotlib.pyplot as plt

# Definimos la carpeta donde vas a guardar tus imagenes originales.
carpeta_imagenes_tfi = Path("imagenes_tfi1")

# Definimos la carpeta donde vas a guardar los resultados finales.
carpeta_salidas_tfi = Path("salidas_tfi1")

# Creamos la carpeta de salida si todavia no existe.
carpeta_salidas_tfi.mkdir(exist_ok=True)

print(f"Carpeta esperada para originales: {carpeta_imagenes_tfi.resolve()}")
print(f"Carpeta esperada para resultados: {carpeta_salidas_tfi.resolve()}")


## Funciones de apoyo

Estas funciones no resuelven el trabajo por vos. Solo ordenan tareas repetitivas de carga, visualizacion y guardado para que puedas concentrarte en el diagnostico y en las decisiones del pipeline.


In [ ]:
def cargar_rgb(ruta):
    """Abre una imagen color y la devuelve en formato RGB."""
    # Leemos la imagen con OpenCV en formato BGR.
    imagen_bgr = cv2.imread(str(ruta), cv2.IMREAD_COLOR)

    # Verificamos que la lectura haya sido correcta.
    if imagen_bgr is None:
        raise FileNotFoundError(f"No se pudo leer la imagen: {ruta}")

    # Convertimos de BGR a RGB para visualizar correctamente con Matplotlib.
    imagen_rgb = cv2.cvtColor(imagen_bgr, cv2.COLOR_BGR2RGB)
    return imagen_rgb


def cargar_gris(ruta):
    """Abre una imagen en escala de grises."""
    # Leemos la imagen directamente como grises.
    imagen_gris = cv2.imread(str(ruta), cv2.IMREAD_GRAYSCALE)

    # Verificamos que la lectura haya sido correcta.
    if imagen_gris is None:
        raise FileNotFoundError(f"No se pudo leer la imagen: {ruta}")

    return imagen_gris


def mostrar_imagen(imagen, titulo="Imagen"):
    """Muestra una sola imagen con un titulo."""
    # Definimos el tamano general de la figura.
    plt.figure(figsize=(7, 6), constrained_layout=True)

    # Elegimos el mapa de color segun la cantidad de dimensiones.
    mapa_de_color = None
    if imagen.ndim == 2:
        mapa_de_color = "gray"

    # Dibujamos la imagen y su titulo.
    plt.imshow(imagen, cmap=mapa_de_color)
    plt.title(titulo, fontweight="bold", loc="left")
    plt.axis("off")
    plt.show()


def mostrar_comparacion(imagen_izquierda, imagen_derecha, titulo_izquierda, titulo_derecha):
    """Muestra una comparacion simple entre dos imagenes."""
    # Creamos una figura con dos ejes lado a lado.
    figura, ejes = plt.subplots(1, 2, figsize=(12, 5), constrained_layout=True)

    # Definimos el mapa de color para la imagen izquierda.
    mapa_izquierda = None
    if imagen_izquierda.ndim == 2:
        mapa_izquierda = "gray"

    # Dibujamos la imagen izquierda.
    ejes[0].imshow(imagen_izquierda, cmap=mapa_izquierda)
    ejes[0].set_title(titulo_izquierda, fontweight="bold", loc="left")
    ejes[0].axis("off")

    # Definimos el mapa de color para la imagen derecha.
    mapa_derecha = None
    if imagen_derecha.ndim == 2:
        mapa_derecha = "gray"

    # Dibujamos la imagen derecha.
    ejes[1].imshow(imagen_derecha, cmap=mapa_derecha)
    ejes[1].set_title(titulo_derecha, fontweight="bold", loc="left")
    ejes[1].axis("off")

    plt.show()


def mostrar_histograma_gris(imagen):
    """Muestra el histograma en grises de una imagen color o gris."""
    # Convertimos a grises si la imagen viene en color.
    imagen_gris = imagen
    if imagen.ndim == 3:
        imagen_gris = cv2.cvtColor(imagen, cv2.COLOR_RGB2GRAY)

    # Calculamos el histograma con NumPy.
    histograma, bordes = np.histogram(imagen_gris.flatten(), bins=256, range=[0, 256])

    # Dibujamos la curva del histograma.
    plt.figure(figsize=(10, 4), constrained_layout=True)
    plt.plot(bordes[:-1], histograma, color="black")
    plt.title("Histograma en escala de grises", fontweight="bold", loc="left")
    plt.xlabel("Intensidad")
    plt.ylabel("Cantidad de pixeles")
    plt.xlim(0, 255)
    plt.grid(alpha=0.3)
    plt.show()


def mostrar_canales_rgb(imagen_rgb):
    """Muestra la imagen color y sus tres canales por separado."""
    # Extraemos los tres canales principales.
    canal_rojo = imagen_rgb[:, :, 0]
    canal_verde = imagen_rgb[:, :, 1]
    canal_azul = imagen_rgb[:, :, 2]

    # Creamos una figura de cuatro paneles.
    figura, ejes = plt.subplots(2, 2, figsize=(10, 8), constrained_layout=True)

    # Mostramos la imagen original.
    ejes[0, 0].imshow(imagen_rgb)
    ejes[0, 0].set_title("Imagen original", fontweight="bold", loc="left")
    ejes[0, 0].axis("off")

    # Mostramos el canal rojo.
    ejes[0, 1].imshow(canal_rojo, cmap="gray")
    ejes[0, 1].set_title("Canal rojo", fontweight="bold", loc="left")
    ejes[0, 1].axis("off")

    # Mostramos el canal verde.
    ejes[1, 0].imshow(canal_verde, cmap="gray")
    ejes[1, 0].set_title("Canal verde", fontweight="bold", loc="left")
    ejes[1, 0].axis("off")

    # Mostramos el canal azul.
    ejes[1, 1].imshow(canal_azul, cmap="gray")
    ejes[1, 1].set_title("Canal azul", fontweight="bold", loc="left")
    ejes[1, 1].axis("off")

    plt.show()


def guardar_rgb(ruta_salida, imagen_rgb):
    """Guarda una imagen RGB en disco."""
    # Convertimos la imagen a BGR para que OpenCV la guarde correctamente.
    imagen_bgr = cv2.cvtColor(imagen_rgb, cv2.COLOR_RGB2BGR)

    # Guardamos la imagen en la ruta indicada.
    cv2.imwrite(str(ruta_salida), imagen_bgr)


def guardar_gris(ruta_salida, imagen_gris):
    """Guarda una imagen en escala de grises."""
    # Guardamos la imagen directamente en la ruta indicada.
    cv2.imwrite(str(ruta_salida), imagen_gris)


## Caso 1. Imagen obtenida mediante camara oscura

En este caso vas a trabajar con una imagen donde suelen aparecer bajo contraste, oscuridad, ruido o fondo excesivo. La pregunta principal no es "que filtro aplico", sino "que problema veo primero y que operacion puede ayudarme".

**Sugerencias tecnicas posibles:** recorte, brillo y contraste, `CLAHE`, suavizado gaussiano o mediana.


In [ ]:
# Escribi aca el nombre de tu archivo para el caso de camara oscura.
nombre_imagen_camara_oscura = ""

# Inicializamos la variable de imagen para evitar errores si todavia no cargaste el archivo.
imagen_camara_oscura_rgb = None

# Cargamos la imagen solo si escribiste un nombre valido.
if nombre_imagen_camara_oscura != "":
    ruta_camara_oscura = carpeta_imagenes_tfi / nombre_imagen_camara_oscura
    imagen_camara_oscura_rgb = cargar_rgb(ruta_camara_oscura)

    # Mostramos la imagen original y su histograma para empezar el diagnostico.
    mostrar_imagen(imagen_camara_oscura_rgb, "Camara oscura: imagen original")
    mostrar_histograma_gris(imagen_camara_oscura_rgb)

# Escribi aca tu diagnostico inicial.
diagnostico_camara_oscura = ""

# Escribi aca tu primera hipotesis de mejora.
hipotesis_camara_oscura = ""

print("Diagnostico inicial:", diagnostico_camara_oscura)
print("Primera hipotesis:", hipotesis_camara_oscura)


### Estrategia 1 para camara oscura

Completa este bloque con una primera via de mejora. Conviene que hagas visible cada paso y que lo nombres con claridad.


In [ ]:
# Partimos de una copia de la imagen original si ya la cargaste.
imagen_camara_estrategia_1 = None
if imagen_camara_oscura_rgb is not None:
    imagen_camara_estrategia_1 = imagen_camara_oscura_rgb.copy()

    # Escribi aca tu pipeline. Algunos caminos posibles son:
    # - recortar la region util
    # - ajustar brillo y contraste con cv2.convertScaleAbs
    # - aplicar CLAHE sobre LAB
    # - suavizar con GaussianBlur o medianBlur
    # imagen_camara_estrategia_1 = ...

    mostrar_comparacion(
        imagen_camara_oscura_rgb,
        imagen_camara_estrategia_1,
        "Original",
        "Camara oscura: estrategia 1"
    )

# Describi brevemente que buscabas con esta estrategia.
observacion_camara_estrategia_1 = ""
print(observacion_camara_estrategia_1)


In [ ]:
# Construimos una segunda estrategia para poder comparar de verdad.
imagen_camara_estrategia_2 = None
if imagen_camara_oscura_rgb is not None:
    imagen_camara_estrategia_2 = imagen_camara_oscura_rgb.copy()

    # Escribi aca una variante distinta a la estrategia 1.
    # imagen_camara_estrategia_2 = ...

    mostrar_comparacion(
        imagen_camara_oscura_rgb,
        imagen_camara_estrategia_2,
        "Original",
        "Camara oscura: estrategia 2"
    )

# Explica en que cambia esta segunda opcion.
observacion_camara_estrategia_2 = ""
print(observacion_camara_estrategia_2)


In [ ]:
# Elegi la version final para este caso.
imagen_camara_final = imagen_camara_estrategia_1

# Justifica por que conservas esta version.
justificacion_camara_final = ""
print(justificacion_camara_final)

# Guardamos la salida final solo si la imagen existe.
if imagen_camara_final is not None:
    ruta_salida_camara = carpeta_salidas_tfi / "camara_oscura_final.png"
    guardar_rgb(ruta_salida_camara, imagen_camara_final)

    if imagen_camara_oscura_rgb is not None:
        mostrar_comparacion(
            imagen_camara_oscura_rgb,
            imagen_camara_final,
            "Camara oscura: antes",
            "Camara oscura: version final"
        )

    print(f"Salida guardada en: {ruta_salida_camara.resolve()}")


## Caso 2. Imagen de medio grafico color

En este caso el problema puede estar en la geometria, en el contraste, en el color o en danos localizados. Antes de intervenir, conviene distinguir si el principal obstaculo es la perspectiva, la cromaticidad, la iluminacion o la presencia de manchas.

**Sugerencias tecnicas posibles:** rectificacion de perspectiva, mejora sobre HSV o LAB, `CLAHE`, suavizado e `inpainting` si hubiera dano localizado.


In [ ]:
# Escribi aca el nombre de tu archivo para el caso de medio grafico color.
nombre_imagen_medio_color = ""

# Inicializamos la variable de imagen para evitar errores si todavia no cargaste el archivo.
imagen_medio_color_rgb = None

# Cargamos la imagen solo si escribiste un nombre valido.
if nombre_imagen_medio_color != "":
    ruta_medio_color = carpeta_imagenes_tfi / nombre_imagen_medio_color
    imagen_medio_color_rgb = cargar_rgb(ruta_medio_color)

    # Mostramos la imagen original y sus canales para orientar el diagnostico.
    mostrar_imagen(imagen_medio_color_rgb, "Medio grafico color: imagen original")
    mostrar_canales_rgb(imagen_medio_color_rgb)

# Escribi aca el problema principal que detectaste.
diagnostico_medio_color = ""

# Escribi aca que queres priorizar en la mejora.
objetivo_medio_color = ""

print("Diagnostico inicial:", diagnostico_medio_color)
print("Objetivo de mejora:", objetivo_medio_color)


In [ ]:
# Partimos de una copia de la imagen original si ya la cargaste.
imagen_medio_color_estrategia_1 = None
if imagen_medio_color_rgb is not None:
    imagen_medio_color_estrategia_1 = imagen_medio_color_rgb.copy()

    # Escribi aca tu primera estrategia. Algunos caminos posibles son:
    # - rectificar perspectiva con cuatro puntos
    # - mejorar contraste en HSV o LAB
    # - aplicar CLAHE o una ecualizacion razonada
    # - usar inpainting si hay dano localizado
    # imagen_medio_color_estrategia_1 = ...

    mostrar_comparacion(
        imagen_medio_color_rgb,
        imagen_medio_color_estrategia_1,
        "Original",
        "Medio color: estrategia 1"
    )

observacion_medio_color_estrategia_1 = ""
print(observacion_medio_color_estrategia_1)


In [ ]:
# Construimos una segunda estrategia para poder discutir una eleccion.
imagen_medio_color_estrategia_2 = None
if imagen_medio_color_rgb is not None:
    imagen_medio_color_estrategia_2 = imagen_medio_color_rgb.copy()

    # Escribi aca una segunda estrategia distinta a la anterior.
    # imagen_medio_color_estrategia_2 = ...

    mostrar_comparacion(
        imagen_medio_color_rgb,
        imagen_medio_color_estrategia_2,
        "Original",
        "Medio color: estrategia 2"
    )

observacion_medio_color_estrategia_2 = ""
print(observacion_medio_color_estrategia_2)

# Elegi la version final para este caso.
imagen_medio_color_final = imagen_medio_color_estrategia_1

# Justifica por que conservas esta version.
justificacion_medio_color_final = ""
print(justificacion_medio_color_final)

# Guardamos la salida final solo si la imagen existe.
if imagen_medio_color_final is not None:
    ruta_salida_medio_color = carpeta_salidas_tfi / "medio_grafico_color_final.png"
    guardar_rgb(ruta_salida_medio_color, imagen_medio_color_final)

    if imagen_medio_color_rgb is not None:
        mostrar_comparacion(
            imagen_medio_color_rgb,
            imagen_medio_color_final,
            "Medio color: antes",
            "Medio color: version final"
        )

    print(f"Salida guardada en: {ruta_salida_medio_color.resolve()}")


## Caso 3. Imagen de medio grafico blanco/negro

En este caso suele importar la legibilidad. El objetivo es separar de manera razonable el fondo y los trazos, sin borrar informacion importante. Por eso conviene pensar si necesitas trabajar en grises, suavizar, umbralizar y limpiar la mascara resultante.

**Sugerencias tecnicas posibles:** conversion a grises, suavizado, umbral global, `Otsu`, `adaptiveThreshold`, apertura o clausura morfologica.


In [ ]:
# Escribi aca el nombre de tu archivo para el caso de medio grafico blanco/negro.
nombre_imagen_medio_bn = ""

# Inicializamos la variable de imagen para evitar errores si todavia no cargaste el archivo.
imagen_medio_bn_gris = None

# Cargamos la imagen solo si escribiste un nombre valido.
if nombre_imagen_medio_bn != "":
    ruta_medio_bn = carpeta_imagenes_tfi / nombre_imagen_medio_bn
    imagen_medio_bn_gris = cargar_gris(ruta_medio_bn)

    # Mostramos la imagen original y su histograma para evaluar contraste y separacion tonal.
    mostrar_imagen(imagen_medio_bn_gris, "Medio grafico blanco/negro: imagen original")
    mostrar_histograma_gris(imagen_medio_bn_gris)

# Escribi aca el problema principal que detectaste.
diagnostico_medio_bn = ""

# Escribi aca que esperas recuperar o mejorar.
objetivo_medio_bn = ""

print("Diagnostico inicial:", diagnostico_medio_bn)
print("Objetivo de mejora:", objetivo_medio_bn)


In [ ]:
# Construimos una primera estrategia para este caso.
imagen_medio_bn_estrategia_1 = None
if imagen_medio_bn_gris is not None:
    imagen_medio_bn_estrategia_1 = imagen_medio_bn_gris.copy()

    # Escribi aca tu primera estrategia. Algunos caminos posibles son:
    # - suavizar antes de umbralizar
    # - probar un umbral global manual
    # - usar Otsu
    # imagen_medio_bn_estrategia_1 = ...

    mostrar_comparacion(
        imagen_medio_bn_gris,
        imagen_medio_bn_estrategia_1,
        "Original",
        "Medio blanco/negro: estrategia 1"
    )

observacion_medio_bn_estrategia_1 = ""
print(observacion_medio_bn_estrategia_1)


In [ ]:
# Construimos una segunda estrategia distinta para comparar criterios.
imagen_medio_bn_estrategia_2 = None
if imagen_medio_bn_gris is not None:
    imagen_medio_bn_estrategia_2 = imagen_medio_bn_gris.copy()

    # Escribi aca una segunda estrategia. Algunos caminos posibles son:
    # - adaptiveThreshold
    # - combinacion de umbralizacion y morfologia
    # imagen_medio_bn_estrategia_2 = ...

    mostrar_comparacion(
        imagen_medio_bn_gris,
        imagen_medio_bn_estrategia_2,
        "Original",
        "Medio blanco/negro: estrategia 2"
    )

observacion_medio_bn_estrategia_2 = ""
print(observacion_medio_bn_estrategia_2)

# Elegi la version final para este caso.
imagen_medio_bn_final = imagen_medio_bn_estrategia_1

# Justifica por que conservas esta version.
justificacion_medio_bn_final = ""
print(justificacion_medio_bn_final)

# Guardamos la salida final solo si la imagen existe.
if imagen_medio_bn_final is not None:
    ruta_salida_medio_bn = carpeta_salidas_tfi / "medio_grafico_bn_final.png"
    guardar_gris(ruta_salida_medio_bn, imagen_medio_bn_final)

    if imagen_medio_bn_gris is not None:
        mostrar_comparacion(
            imagen_medio_bn_gris,
            imagen_medio_bn_final,
            "Medio blanco/negro: antes",
            "Medio blanco/negro: version final"
        )

    print(f"Salida guardada en: {ruta_salida_medio_bn.resolve()}")


## Cierre comparativo

Ahora te toca salir del caso puntual y mirar el integrador como conjunto. La pregunta no es solo que funciono, sino tambien **por que convino una estrategia en un caso y no en otro**.


In [ ]:
# Construimos una tabla de comparacion final entre los tres casos.
casos = [
    "Camara oscura",
    "Medio grafico color",
    "Medio grafico blanco/negro"
]

problemas_principales = [
    diagnostico_camara_oscura,
    diagnostico_medio_color,
    diagnostico_medio_bn
]

estrategias_finales = [
    justificacion_camara_final,
    justificacion_medio_color_final,
    justificacion_medio_bn_final
]

limites_detectados = [
    "",
    "",
    ""
]

tabla_final = pd.DataFrame({
    "caso": casos,
    "problema_principal": problemas_principales,
    "decision_final": estrategias_finales,
    "limites": limites_detectados
})

display(tabla_final)


## Reflexion final

Respondan con sus palabras:

1. ¿Que decision tecnica les resulto mas dificil de justificar y por que?
2. ¿En que caso la mejora fue mas clara y en cual fue mas limitada?
3. ¿Que aprendieron sobre la relacion entre tipo de degradacion y tipo de operacion?
4. ¿Que harian distinto si tuvieran una segunda iteracion de trabajo?

## Lista de control antes de entregar

- Resolvi los tres casos obligatorios.
- Mostre la imagen original en cada caso.
- Compare al menos dos estrategias por caso.
- Justifique la eleccion final de cada pipeline.
- Guarde las tres imagenes procesadas.
- Complete la tabla final de comparacion.
- Registre el uso de IA de manera breve y critica.
- Escribi una reflexion final con limites del metodo.

## Defensa oral breve

En la defensa cada integrante tiene que poder explicar el problema principal de cada caso, la razon de la estrategia elegida y el limite mas importante de la restauracion realizada.
